In [1]:
# This notebook use the jsonl data shared by the instructor, and create a new database

In [2]:

import json
import time

import psycopg
from psycopg import OperationalError

In [ ]:
# def load_data(filename):
#     """
#     Loads cleaned data from a JSON file.
#     """
#     with open(filename, "r", encoding="utf-8") as f:
#         data = json.load(f)
#     return data

In [ ]:
# directory = '/Users/jennifer/Documents/software_concept_python_class/jhu_software_concepts/Module_2/'

# applicant_data= load_data(directory+ "applicant_data_v2.json")
# len(applicant_data)

41000

In [5]:
def load_data_jsonl(filename):
    """
    Loads data from a JSONL file. Each line is one JSON object.
    """
    data = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# back up version of the data
applicant_data= load_data_jsonl("llm_extend_applicant_data_run.jsonl")
len(applicant_data)

36000

In [6]:
type(applicant_data)

list

In [7]:
# inspect the data

print(applicant_data[0])


{'program': 'Environmental Economics, Stockholm University', 'university_raw': 'Stockholm University', 'program_raw': 'Environmental Economics', 'comments': '', 'date_added': 'May 31, 2026', 'url': 'https://www.thegradcafe.com/result/1020288', 'status': 'Accepted on May 31', 'acceptance_date': 'May 31', 'rejection_date': None, 'term': 'Fall 2026', 'US/International': 'International', 'Degree': 'PhD', 'GPA': None, 'GRE': None, 'GRE V': None, 'GRE AW': None, 'llm-generated-program': 'Environmental Economics', 'llm-generated-university': 'Stockholm University'}


In [8]:
print(applicant_data[0].keys()) 

dict_keys(['program', 'university_raw', 'program_raw', 'comments', 'date_added', 'url', 'status', 'acceptance_date', 'rejection_date', 'term', 'US/International', 'Degree', 'GPA', 'GRE', 'GRE V', 'GRE AW', 'llm-generated-program', 'llm-generated-university'])


In [9]:
# Create connection to system database
def create_system_connection(user, password):
    """Connect to postgres system database"""
    try:
        conn = psycopg.connect(
            dbname="postgres",
            user=user,
            password=password,
            host="localhost"
        )
        conn.autocommit = True
        return conn
    except OperationalError as e:
        print(f"Connection failed: {e}")
        return None

In [11]:
# Create new database
def create_database(conn, db_name):
    """Create a new database"""
    try:
        cur = conn.cursor()
        cur.execute(f"CREATE DATABASE {db_name};")
        cur.close()
        print(f"Database '{db_name}' created successfully")
    except OperationalError as e:
        print(f"Database creation failed: {e}")

In [12]:
# created the database gradcafe_db_v2

system_conn = create_system_connection("postgres", "181818")
create_database(system_conn, "gradcafe_db_v2")
system_conn.close()

Database 'gradcafe_db_v2' created successfully


In [13]:
def create_db_connection(db_name, db_user, db_password, db_host, db_port):

    """
    create conenction to my db
    """

    connection = None
    try:
        connection = psycopg.connect(
            dbname=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
        )
        print("Connection to PostgreSQL DB successful")
    except OperationalError as e:
        print(f"The error '{e}' occurred")
    return connection

In [14]:
connection = create_db_connection("gradcafe_db_v2", "postgres", "181818", "127.0.0.1", "5432")

Connection to PostgreSQL DB successful


In [15]:
def execute_query(connection, query):
    connection.autocommit = True
    cursor = connection.cursor()
    try:
        cursor.execute(query)
        # print("Query executed successfully")
    except OperationalError as e:
        print(f"The error '{e}' occurred")

#### create table

In [16]:
create_users_table = """
CREATE TABLE IF NOT EXISTS applicants (
p_id SERIAL PRIMARY KEY,
program	TEXT,
comments TEXT,
date_added date,
url	TEXT,
status TEXT,
term TEXT,
us_or_international	TEXT,
gpa	FLOAT,
gre	FLOAT,
gre_v FLOAT,
gre_aw FLOAT,
degree TEXT,
llm_generated_program TEXT,
llm_generated_university TEXT
)
"""

execute_query(connection, create_users_table)

#### insert records

In [18]:
# check datatype of the json data file
for key, value in applicant_data[0].items():
    print(key, type(value))

program <class 'str'>
university_raw <class 'str'>
program_raw <class 'str'>
comments <class 'str'>
date_added <class 'str'>
url <class 'str'>
status <class 'str'>
acceptance_date <class 'str'>
rejection_date <class 'NoneType'>
term <class 'str'>
US/International <class 'str'>
Degree <class 'str'>
GPA <class 'NoneType'>
GRE <class 'NoneType'>
GRE V <class 'NoneType'>
GRE AW <class 'NoneType'>
llm-generated-program <class 'str'>
llm-generated-university <class 'str'>


In [19]:
def format_text(value):
    """
    The current value: 
    1.if the value is None,it returns "Null"
    2. replace ' with '' (SQL needs it)
    3. the whole thing quoted with ' ' (SQL needs it)
    """
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def format_float(value):
    """
    1.if the value is None,it returns "Null"
    2. extract float from json, then convert to string for SQL insertion
    """
    if value is None or value == "":
        return "NULL"
    return str(float(value))


# def create_program(applicant):
#     university = applicant.get("llm-generated-university")
#     program_name = applicant.get("llm-generated-program")

#     if university is None and program_name is None:
#         return "NULL"

#     if university is None:
#         full_program = program_name
#     elif program_name is None:
#         full_program = university
#     else:
#         full_program = program_name + ", " + university

#     return format_text(full_program)


def insert_applicants(connection, applicant_data):
    for applicant in applicant_data:
        insert_query = f"""
        INSERT INTO applicants (
            program,
            comments,
            date_added,
            url,
            status,
            term,
            us_or_international,
            gpa,
            gre,
            gre_v,
            gre_aw,
            degree,
            llm_generated_program,
            llm_generated_university
        )
        VALUES (
            {format_text(applicant.get("program"))},
            {format_text(applicant.get("comments"))},
            {format_text(applicant.get("date_added"))},
            {format_text(applicant.get("url"))},
            {format_text(applicant.get("status"))},
            {format_text(applicant.get("term"))},
            {format_text(applicant.get("US/International"))},
            {format_float(applicant.get("GPA"))},
            {format_float(applicant.get("GRE"))},
            {format_float(applicant.get("GRE V"))},
            {format_float(applicant.get("GRE AW"))},
            {format_text(applicant.get("Degree"))},
            {format_text(applicant.get("llm-generated-program"))},
            {format_text(applicant.get("llm-generated-university"))}
        );
        """

        execute_query(connection, insert_query)


insert_applicants(connection, applicant_data)

#### Read query

In [20]:
# Note: the two functions are different: 
# execute_query ---- runs query, does not fetch data
# execute_read_query ---- runs SELECT query, fetches and returns data

def execute_read_query(connection, query):
    cursor = connection.cursor()
    result = None
    try:
        cursor.execute(query)
        result = cursor.fetchall()
        return result
    except OperationalError as e:
        print(f"The error '{e}' occurred")


select_applicants = "SELECT * FROM applicants limit 10"
applicants = execute_read_query(connection, select_applicants)

for applicant in applicants:
    print(applicant)

(1, 'Environmental Economics, Stockholm University', '', datetime.date(2026, 5, 31), 'https://www.thegradcafe.com/result/1020288', 'Accepted on May 31', 'Fall 2026', 'International', None, None, None, None, 'PhD', 'Environmental Economics', 'Stockholm University')
(2, 'Global Health, Meharry Medical College', '', datetime.date(2026, 5, 30), 'https://www.thegradcafe.com/result/1020287', 'Accepted on May 29', 'Fall 2026', 'American', 3.9, None, None, None, 'PhD', 'Global Health', 'Meharry Medical College')
(3, 'Public Policy, University of Massachusetts', '', datetime.date(2026, 5, 29), 'https://www.thegradcafe.com/result/1020286', 'Wait Listed on May 29', 'Fall 2026', 'International', None, None, None, None, 'PhD', 'Public Policy', 'University of Massachusetts')
(4, 'Engineering Science, University of Oxford', "Applying as a (home student) Oxford undergrad. Didn't receive the offer until super late...!", datetime.date(2026, 5, 29), 'https://www.thegradcafe.com/result/1020285', 'Accepted

In [21]:
total_applicants = "SELECT count(*) FROM applicants"
tot = execute_read_query(connection, total_applicants)
print(tot)

[(36000,)]


#### Q1: How many entries do you have in your database who have applied for Fall 2026?

In [22]:
# first check why type of data in term
query1a = "SELECT distinct term FROM applicants"
query1a_output = execute_read_query(connection, query1a)
print(query1a_output)

[('Spring 2025',), ('Fall 2025',), ('Fall 2024',), ('Spring 2026',), ('Fall 2026',)]


In [23]:
query1 = "SELECT count(*) FROM applicants where term = 'Fall 2026'"
query1_output = execute_read_query(connection, query1)
print(query1_output)

[(33052,)]


#### Q2: What percentage of entries are from international students (not American or Other) (to two decimal places)?

In [25]:
# use triple quoted string for nice layout query!
query2 = """
SELECT
    ROUND(100.0 * SUM(CASE WHEN us_or_international = 'International' THEN 1
                           ELSE 0
                      END
                      ) / COUNT(*),
        2
    ) AS international_pct
FROM applicants;
"""

query2_output = execute_read_query(connection, query2)
print(query2_output)

[(Decimal('45.22'),)]


In [26]:
# double check query2 results
query2_check = """
SELECT distinct us_or_international, count(*)
from applicants
group by us_or_international;
"""

query2_chk_output = execute_read_query(connection, query2_check)
print(query2_chk_output)



[('American', 18933), ('International', 16279), (None, 788)]


#### Q3: What is the average GPA, GRE, GRE V, GRE AW of applicants who provide these metrics?

In [27]:
# query3 = """
# SELECT
#     ROUND(AVG(gpa), 2) AS avg_gpa,
#     ROUND(AVG(gre), 2) AS avg_gre,
#     ROUND(AVG(gre_v),2) AS avg_gre_v,
#     ROUND(AVG(gre_aw),2) AS avg_gre_aw
# FROM applicants;
# """
query3 = """
SELECT
    ROUND(AVG(gpa)::numeric, 2) AS avg_gpa,
    ROUND(AVG(gre)::numeric, 2) AS avg_gre,
    ROUND(AVG(gre_v)::numeric, 2) AS avg_gre_v,
    ROUND(AVG(gre_aw)::numeric, 2) AS avg_gre_aw
FROM applicants;
"""

query3_output = execute_read_query(connection, query3)
print(query3_output)

[(Decimal('3.78'), Decimal('262.00'), Decimal('161.54'), Decimal('9.30'))]


In [28]:

avg_gpa, avg_gre, avg_gre_v, avg_gre_aw = query3_output[0]

print("Average GPA:", avg_gpa)
print("Average GRE:", avg_gre)
print("Average GRE V:", avg_gre_v)
print("Average GRE AW:", avg_gre_aw)

Average GPA: 3.78
Average GRE: 262.00
Average GRE V: 161.54
Average GRE AW: 9.30


#### Q4: What is their average GPA of American students in Fall 2026?

In [29]:
query4 = """
SELECT
    ROUND(AVG(gpa)::numeric, 2) AS avg_gpa
FROM applicants
where us_or_international = 'American' and term = 'Fall 2026';
"""

query4_output = execute_read_query(connection, query4)
print(query4_output)

[(Decimal('3.79'),)]


#### Q5: What percent of entries for Fall 2026 are Acceptances (to two decimal places)?

In [61]:
query5a = """
SELECT
   distinct status
FROM applicants
where term = 'Fall 2026';
"""

query5a_output = execute_read_query(connection, query5a)
print(query5a_output)

[('Accepted on Dec 31',), ('Interview on Jan 14',), ('Accepted on Feb 26',), ('Accepted on Dec 13',), ('Wait Listed on Mar 17',), ('Rejected on Jan 03',), ('Accepted on Feb 06',), ('Accepted on Oct 14',), ('Rejected on Feb 17',), ('Wait Listed on Mar 21',), ('Accepted on Apr 04',), ('Rejected on Dec 05',), ('Rejected on Apr 30',), ('Accepted on Jun 14',), ('Wait Listed on Jan 20',), ('Accepted on Feb 18',), ('Interview on Mar 25',), ('Rejected on Apr 05',), ('Rejected on Jan 08',), ('Interview on Jan 26',), ('Wait Listed on Mar 01',), ('Interview on Dec 10',), ('Rejected on Dec 11',), ('Accepted on May 07',), ('Interview on Dec 09',), ('Accepted on Aug 06',), ('Wait Listed on Jan 05',), ('Rejected on Mar 16',), ('Accepted on Nov 05',), ('Interview on Feb 24',), ('Interview on Sep 02',), ('Interview on Dec 03',), ('Rejected on Feb 27',), ('Wait Listed on Jan 02',), ('Interview on Mar 23',), ('Accepted on Dec 04',), ('Interview on Nov 04',), ('Accepted on Feb 13',), ('Rejected on Oct 04'

In [63]:
query5 = """
SELECT
    ROUND(
        100.0 * SUM(
            CASE WHEN status ILIKE '%Accepted%' THEN 1
                 ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS fall_2026_accept_pct
FROM applicants
WHERE term = 'Fall 2026';
"""

query5_output = execute_read_query(connection, query5)
print(query5_output)

[(Decimal('35.84'),)]


#### Q6:What is the average GPA of applicants who applied for Fall 2026 who are Acceptances?

In [32]:
query6 = """
SELECT
   avg(gpa)
FROM applicants
where term = 'Fall 2026' and status = 'Accepted';
"""

query6_output = execute_read_query(connection, query6)
print(query6_output)

[(None,)]


#### Q7: How many entries are from applicants who applied to JHU for a masters degrees in Computer Science?

In [36]:
query7a = """
SELECT
   distinct degree
FROM applicants;
"""

query7a_output = execute_read_query(connection, query7a)
print(query7a_output)

[('PhD',), ('Other',), ('MFA',), ('EdD',), ('JD',), ('Masters',), ('PsyD',), ('MBA',)]


In [ ]:
# this test print the records found
query7b = """
SELECT *
FROM applicants
WHERE degree = 'Masters'
  AND (
      "llm_generated_university" ILIKE '%Johns Hopkins%'
      OR "llm_generated_university" ILIKE '%John Hopkins%'
      OR "llm_generated_university" ILIKE '%JHU%'
  )
  AND "llm_generated_program" ILIKE '%computer science%';
"""

query7b_output = execute_read_query(connection, query7b)
print(query7b_output)

[(1800, 'Computer Science, Johns Hopkins University', '', datetime.date(2026, 4, 4), 'https://www.thegradcafe.com/result/1018487', 'Rejected on Apr 03', 'Fall 2026', 'International', 3.67, None, None, None, 'Masters', 'Computer Science', 'Johns Hopkins University'), (2286, 'Computer Science, Johns Hopkins University', '', datetime.date(2026, 3, 31), 'https://www.thegradcafe.com/result/1018001', 'Accepted on Mar 31', 'Fall 2026', 'International', 3.67, None, None, None, 'Masters', 'Computer Science', 'Johns Hopkins University'), (2307, 'Computer Science, Johns Hopkins University', '', datetime.date(2026, 3, 31), 'https://www.thegradcafe.com/result/1017980', 'Accepted on Mar 31', 'Fall 2026', 'International', None, None, None, None, 'Masters', 'Computer Science', 'Johns Hopkins University'), (2319, 'Computer Science, Johns Hopkins University', 'Checked the portal and receive status update', datetime.date(2026, 3, 31), 'https://www.thegradcafe.com/result/1017968', 'Accepted on Mar 31', 'F

In [55]:
query7 = """
SELECT count(*)
FROM applicants
WHERE degree = 'Masters'
  AND (
      program ILIKE '%Johns Hopkins%'
      OR program ILIKE '%John Hopkins%'
      OR program ILIKE '%JHU%'
  )
  AND program ILIKE '%computer science%';
"""
# ILIKE is case insensitive
query7_output = execute_read_query(connection, query7)
print(query7_output)

[(14,)]


#### Q8: How many entries from 2026 are acceptances from applicants who applied to Georgetown University, MIT, Stanford University, or Carnegie Mellon University for a PhD in Computer Science?

In [56]:
# query8 = """
# SELECT COUNT(*)
# FROM applicants
# WHERE status = 'Accepted'
#   AND degree = 'PhD'
#   AND llm_generated_program ILIKE '%computer science%'
#   AND EXTRACT(YEAR FROM date_added) = 2026
#   AND (
#       llm_generated_university = 'Georgetown University'
#       OR llm_generated_university = 'Stanford University'
#       OR llm_generated_university = 'Carnegie Mellon University'
#       OR llm_generated_university = 'Massachusetts Institute of Technology'
#       OR llm_generated_university = 'MIT'
#   );
# """

query8 = """
SELECT COUNT(*)
FROM applicants
WHERE status = 'Accepted'
  AND degree = 'PhD'
  AND program ILIKE '%computer science%'
  AND EXTRACT(YEAR FROM date_added) = 2026
  AND (
      program ILIKE '%Georgetown%'
      OR program ILIKE '%Stanford%'
      OR program ILIKE '%Carnegie Mellon%'
      OR program ILIKE '%Massachusetts Institute of Technology%'
      OR program ILIKE '%MIT%'
  );
"""

query8_output = execute_read_query(connection, query8)
print(query8_output)

[(0,)]


#### Q9: Do you numbers for question 8 change if you use LLM Generated Fields (rather than your downloaded fields)?

In [59]:
query9 = """
SELECT COUNT(*)
FROM applicants
WHERE status = 'Accepted'
  AND degree = 'PhD'
  AND llm_generated_program ILIKE '%computer science%'
  AND EXTRACT(YEAR FROM date_added) = 2026
  AND (
      llm_generated_university ILIKE '%Georgetown%'
      OR llm_generated_university ILIKE '%Stanford%'
      OR llm_generated_university ILIKE '%Carnegie Mellon%'
      OR llm_generated_university ILIKE '%Massachusetts Institute of Technology%'
      OR llm_generated_university ILIKE '%MIT%'
  );
"""

query9_output = execute_read_query(connection, query9)
print(query9_output)

[(0,)]


#### Q10: How many entries for students applied for Fall 2026 term for Stanford Univerity Masters program per acceptance status? 

In [60]:
query10 = """
SELECT
    status,
    COUNT(*) AS num_entries
FROM applicants
WHERE term = 'Fall 2026'
  AND degree = 'Masters'
  AND llm_generated_university ILIKE '%Stanford%'
GROUP BY status
ORDER BY num_entries DESC;
"""

query10_output = execute_read_query(connection, query10)
print(query10_output)

[('Rejected on Feb 27', 22), ('Accepted on Mar 13', 10), ('Accepted on Feb 13', 7), ('Rejected on Dec 17', 7), ('Rejected on Mar 20', 7), ('Accepted on Feb 27', 7), ('Rejected on Mar 25', 7), ('Accepted on Feb 25', 5), ('Rejected on Mar 13', 5), ('Rejected on Mar 03', 4), ('Accepted on Mar 25', 4), ('Rejected on Mar 24', 3), ('Accepted on Mar 02', 2), ('Accepted on Mar 06', 2), ('Accepted on Mar 19', 2), ('Rejected on Mar 26', 2), ('Interview on Feb 06', 2), ('Rejected on Feb 24', 2), ('Rejected on Mar 31', 1), ('Wait Listed on Feb 27', 1), ('Accepted on Dec 17', 1), ('Wait Listed on Mar 24', 1), ('Accepted on Feb 12', 1), ('Accepted on Feb 19', 1), ('Accepted on Feb 24', 1), ('Accepted on Feb 26', 1), ('Accepted on Jan 23', 1), ('Accepted on Jan 29', 1), ('Accepted on Mar 16', 1), ('Accepted on Mar 24', 1), ('Accepted on Mar 31', 1), ('Rejected on Feb 06', 1), ('Rejected on Feb 26', 1), ('Rejected on Feb 28', 1), ('Rejected on Jan 23', 1), ('Rejected on Jan 29', 1), ('Rejected on Mar 